# Online Payment Fraud Detection — Exploratory Data Analysis (EDA)

**Course**: Machine Learning with Business Data (MLBD)  
**Dataset**: PaySim Synthetic Financial Transactions (`new_data.csv`)  
**Target**: `isFraud` (1 = Fraudulent, 0 = Legitimate)

---
### Objectives:
1. Inspect data schema, types, missing values, and integrity.
2. Analyze class imbalance in online financial transactions.
3. Discover which transaction types contain fraudulent activities.
4. Examine behavioral balance anomalies and temporal patterns.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
DATA_PATH = os.path.join('..', 'data', 'raw', 'new_data.csv')

print(f"Loading dataset from {DATA_PATH}...")
df = pd.read_csv(DATA_PATH)
print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 1. Data Integrity and Missing Values

In [ ]:
print("Missing Values per Column:")
print(df.isnull().sum())
print(f"\nDuplicate Rows: {df.duplicated().sum():,}")

## 2. Target Class Distribution (Extreme Imbalance)

In [ ]:
counts = df['isFraud'].value_counts()
print("Class Counts:")
print(counts)
fraud_rate = (counts[1] / len(df)) * 100
print(f"\nFraud Incident Rate: {fraud_rate:.4f}%")
print(f"Imbalance Ratio: {counts[0]/counts[1]:.2f} : 1")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Legitimate (0)', 'Fraud (1)'], counts.values, color=['#2b5c8f', '#d9534f'])
ax.set_yscale('log')
ax.set_ylabel('Transaction Count (Log Scale)')
ax.set_title('Transaction Volume by Class (Log Scale)')
plt.show()

## 3. Transaction Types and Fraud Incidence

A crucial domain finding in PaySim is discovering **where fraud actually occurs**.

In [ ]:
fraud_by_type = df.groupby('type')['isFraud'].agg(['count', 'sum'])
fraud_by_type['fraud_rate_%'] = (fraud_by_type['sum'] / fraud_by_type['count']) * 100
fraud_by_type.columns = ['Total_Transactions', 'Fraud_Cases', 'Fraud_Rate_%']
print(fraud_by_type)

# Notice: Fraud is strictly 0 for PAYMENT, CASH_IN, and DEBIT!
# Fraud ONLY occurs in TRANSFER and CASH_OUT!

## 4. Balance Discrepancies and Behavioral Anomalies

In legitimate transactions, `newbalanceOrig = oldbalanceOrg - amount`.
In fraudulent transfers, perpetrators often empty the sender's account entirely.

In [ ]:
df['orig_emptied'] = ((df['newbalanceOrig'] == 0) & (df['oldbalanceOrg'] > 0)).astype(int)
emptied_fraud = df.groupby('isFraud')['orig_emptied'].mean() * 100
print("Percentage of transactions where origin balance was emptied to 0.0:")
print(f"Legitimate: {emptied_fraud[0]:.2f}%")
print(f"Fraudulent: {emptied_fraud[1]:.2f}%")

## 5. Summary Insights for Modeling

1. **Severe Imbalance (~0.08% to 0.13%)**: Standard accuracy is deceptive; models must be evaluated using **PR-AUC (AUPRC)**, Recall, and Precision.
2. **Specific Vectors**: Machine learning models can focus on `TRANSFER` and `CASH_OUT` transaction types where fraud actually manifests.
3. **Engineered Discrepancies**: Features capturing balance changes (`errorBalanceOrig`, `orig_emptied`) provide the highest predictive signal across decision trees and neural networks.